<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 8.5 - Prompting Large Language Models

In this lab we will practise prompting with a few Large Language Models (LLMs) using Groq (not to be confused with Grok). Groq is a platform that provides access to their custom-built AI hardware via APIs, allowing users to run open-source models such as Llama.

We shall see that while LLMs are powerful tools, how you ask a question or frame a task can dramatically influence the results obtained.

## Set-up

Step 1: Sign up for a free Groq account at https://console.groq.com/home .

Step 2: Create a new API key at https://console.groq.com/keys. Copy-paste it into an empty text file called 'groq_key.txt'.

Running the next cell will then read in this key and assign it to the variable `groq_key`.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving groq_key.txt to groq_key.txt


In [2]:
groqfilename = r'groq_key.txt' # this file contains a single line containing your Groq API key only
try:
    with open(groqfilename, 'r') as f:
        groq_key = f.read().strip()
except FileNotFoundError:
    print("'%s' file not found" % filename)

In [3]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 3.3 MB/s eta 0:00:00


In [4]:
from groq import Groq
import requests
import pandas as pd
from IPython.display import Markdown

First create an instance of the Groq client:

In [5]:
client = Groq(api_key=groq_key)

The following code shows what models are currently accessible through Groq. `context_window` refers to the size of memory (in tokens) during a session and `max_completion_tokens` is the maximum number of tokens that are generated in an output.

In [6]:
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {groq_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

pd.DataFrame(response.json()['data']).sort_values(['created'], ascending=False)

,id,object,created,owned_by,active,context_window,public_apps,max_completion_tokens
7,meta-llama/llama-prompt-guard-2-86m,model,1748632165,Meta,True,512,None,512
13,meta-llama/llama-prompt-guard-2-22m,model,1748632101,Meta,True,512,None,512
20,qwen/qwen3-32b,model,1748396646,Alibaba Cloud,True,131072,None,40960
8,meta-llama/llama-guard-4-12b,model,1746743847,Meta,True,131072,None,1024
21,meta-llama/llama-4-maverick-17b-128e-instruct,model,1743877158,Meta,True,131072,None,8192
4,meta-llama/llama-4-scout-17b-16e-instruct,model,1743874824,Meta,True,131072,None,8192
19,compound-beta-mini,model,1742953279,Groq,True,131072,None,8192
2,qwen-qwq-32b,model,1741214760,Alibaba Cloud,True,131072,None,131072
9,compound-beta,model,1740880017,Groq,True,131072,None,8192
3,playai-tts-arabic,model,1740682783,PlayAI,True,8192,None,8192


The Groq client object enables interaction with the Groq REST API and a chat completion request is made via the client.chat.completions.create method.

The most important arguments of the client.chat.completions.create method are the following:
* messages: a list of messages (dictionary form) that make up the conversation to date
* model: a string indicating which model to use (see [list of models](https://console.groq.com/docs/models))
* max_completion_tokens: the maximum number of tokens that are generated in the chat completion
* response_format: setting this to `{ "type": "json_object" }` enables JSON output
* seed: sample deterministically as best as possible, though identical outputs each time are not guaranteed
* temperature: between 0 and 2 where higher values like 0.8 make the output more random (creative) and values like 0.2 are more focused and deterministic


In [7]:
help(client.chat.completions.create)

Help on method create in module groq.resources.chat.completions:

create(*, messages: 'Iterable[ChatCompletionMessageParam]', model: "Union[str, Literal['gemma2-9b-it', 'llama-3.3-70b-versatile', 'llama-3.1-8b-instant', 'llama-guard-3-8b', 'llama3-70b-8192', 'llama3-8b-8192']]", exclude_domains: 'Optional[List[str]] | NotGiven' = NOT_GIVEN, frequency_penalty: 'Optional[float] | NotGiven' = NOT_GIVEN, function_call: 'Optional[completion_create_params.FunctionCall] | NotGiven' = NOT_GIVEN, functions: 'Optional[Iterable[completion_create_params.Function]] | NotGiven' = NOT_GIVEN, include_domains: 'Optional[List[str]] | NotGiven' = NOT_GIVEN, logit_bias: 'Optional[Dict[str, int]] | NotGiven' = NOT_GIVEN, logprobs: 'Optional[bool] | NotGiven' = NOT_GIVEN, max_completion_tokens: 'Optional[int] | NotGiven' = NOT_GIVEN, max_tokens: 'Optional[int] | NotGiven' = NOT_GIVEN, metadata: 'Optional[Dict[str, str]] | NotGiven' = NOT_GIVEN, n: 'Optional[int] | NotGiven' = NOT_GIVEN, parallel_tool_calls:

As a first example, note how the messages input is given as a list of a dictionaries with `role` and `content` keys. This is in a ChatML format recognised by many LLMs.

In [8]:
chat_completion = client.chat.completions.create(
    messages=[
        {   "role": "system", # sets the persona of the model
            "content": "You are a helpful assistant."
        },
        {
            "role": "user", # what the user wants the assistant to do
            "content": "Explain briefly how large language models work",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Large language models are a type of artificial intelligence (AI) that process and understand human language. They work by:

1. **Training**: Being trained on vast amounts of text data, such as books, articles, and conversations.
2. **Pattern recognition**: Recognizing patterns and relationships in language, such as grammar, syntax, and semantics.
3. **Prediction**: Using these patterns to predict the next word or character in a sequence, based on context and probability.

When you ask a question or provide input, the model:

1. **Tokenizes**: Breaks down your input into individual words or tokens.
2. **Analyzes**: Analyzes the tokens to understand their meaning and context.
3. **Generates**: Generates a response based on the patterns and relationships learned during training.

This process allows large language models to generate human-like text, answer questions, and even engage in conversations.


The output is in Markdown format so the following line formats this text.

In [9]:
Markdown(chat_completion.choices[0].message.content)

Large language models are a type of artificial intelligence (AI) that process and understand human language. They work by:

1. **Training**: Being trained on vast amounts of text data, such as books, articles, and conversations.
2. **Pattern recognition**: Recognizing patterns and relationships in language, such as grammar, syntax, and semantics.
3. **Prediction**: Using these patterns to predict the next word or character in a sequence, based on context and probability.

When you ask a question or provide input, the model:

1. **Tokenizes**: Breaks down your input into individual words or tokens.
2. **Analyzes**: Analyzes the tokens to understand their meaning and context.
3. **Generates**: Generates a response based on the patterns and relationships learned during training.

This process allows large language models to generate human-like text, answer questions, and even engage in conversations.

## Text summarisation

We start with a llama3-8b-8192, a model using just over 8 billion parameters with at most 8192 tokens produced as output.

Here is an article to be summarised from the [cnn_dailymail](https://huggingface.co/datasets/cnn_dailymail) dataset:

In [10]:
story = """
SAN FRANCISCO, California (CNN) -- A magnitude 4.2 earthquake shook the San Francisco area Friday at 4:42 a.m. PT (7:42 a.m. ET), the U.S. Geological Survey reported. The quake left about 2,000 customers without power, said David Eisenhower, a spokesman for Pacific Gas and Light. Under the USGS classification, a magnitude 4.2 earthquake is considered "light," which it says usually causes minimal damage. "We had quite a spike in calls, mostly calls of inquiry, none of any injury, none of any damage that was reported," said Capt. Al Casciato of the San Francisco police. "It was fairly mild." Watch police describe concerned calls immediately after the quake » . The quake was centered about two miles east-northeast of Oakland, at a depth of 3.6 miles, the USGS said. Oakland is just east of San Francisco, across San Francisco Bay. An Oakland police dispatcher told CNN the quake set off alarms at people's homes. The shaking lasted about 50 seconds, said CNN meteorologist Chad Myers. According to the USGS, magnitude 4.2 quakes are felt indoors and may break dishes and windows and overturn unstable objects. Pendulum clocks may stop.
"""

**Exercise:**
Summarise the story text using the following three prompts. Use the format given above but here there is no need to set the persona (i.e. only include one dictionary in the messages list when calling `client.chat.completions.create`.) Comment on any differences.

1) "Summarise the following article in 3 sentences."

2) "Give me a TL;DR of this text."

3) "What's the key takeaway here?"

In [13]:
prompts = ["Summarise the following article in 3 sentences. ", "Give me a TL;DR of this text. ", "What's the key takeaway here?"]
#content will be p + story for p in prompts
def get_summary(prompt):
    completion = client.chat.completions.create(
        messages=[{
            "role": "user",
            "content": f"{prompt}\n\n{story}"
        }],
        model="llama3-8b-8192",
        max_tokens=512,
        temperature=0.2
    )
    return completion.choices[0].message.content.strip()

# 1. Summarise in 3 sentences
summary_3_sentences = get_summary("Summarise the following article in 3 sentences.")

# 2. TL;DR
summary_tldr = get_summary("Give me a TL;DR of this text.")

# 3. Key takeaway
summary_takeaway = get_summary("What's the key takeaway here?")

# Display all results
print("=== Summary (3 Sentences) ===")
print(summary_3_sentences)
print("\n=== TL;DR ===")
print(summary_tldr)
print("\n=== Key Takeaway ===")
print(summary_takeaway)
# ANSWER


=== Summary (3 Sentences) ===
Here is a summary of the article in 3 sentences:

A magnitude 4.2 earthquake struck the San Francisco area at 4:42 a.m. PT on Friday, causing minimal damage and no reported injuries. The quake left around 2,000 customers without power, but authorities reported no significant damage or injuries. The earthquake was centered about two miles east-northeast of Oakland and was felt for about 50 seconds, causing some alarm and concern among residents.

=== TL;DR ===
A 4.2 magnitude earthquake struck the San Francisco area at 4:42am, causing about 2,000 power outages but no reported injuries or significant damage. The quake was centered near Oakland and lasted about 50 seconds, with some residents reporting alarms going off at their homes.

=== Key Takeaway ===
The key takeaway is that a magnitude 4.2 earthquake struck the San Francisco area, causing minimal damage and no reported injuries, but leaving about 2,000 customers without power.


Run the above code again below and note that the answers may differ. This is due to the probabilistic nature of LLM token generation.

In [14]:
# ANSWER
def get_summary(prompt):
    completion = client.chat.completions.create(
        messages=[{
            "role": "user",
            "content": f"{prompt}\n\n{story}"
        }],
        model="llama3-8b-8192",
        max_tokens=512,
        temperature=0.2
    )
    return completion.choices[0].message.content.strip()

# 1. Summarise in 3 sentences
summary_3_sentences = get_summary("Summarise the following article in 3 sentences.")

# 2. TL;DR
summary_tldr = get_summary("Give me a TL;DR of this text.")

# 3. Key takeaway
summary_takeaway = get_summary("What's the key takeaway here?")

# Display all results
print("=== Summary (3 Sentences) ===")
print(summary_3_sentences)
print("\n=== TL;DR ===")
print(summary_tldr)
print("\n=== Key Takeaway ===")
print(summary_takeaway)

=== Summary (3 Sentences) ===
Here is a summary of the article in 3 sentences:

A magnitude 4.2 earthquake struck the San Francisco area at 4:42 a.m. PT on Friday, causing minimal damage and leaving about 2,000 customers without power. The quake was considered "light" by the USGS and was felt indoors, with reports of alarms going off at homes and some minor disruptions. The shaking lasted about 50 seconds and no injuries or significant damage were reported, with authorities describing the quake as "fairly mild".

=== TL;DR ===
A magnitude 4.2 earthquake struck the San Francisco area at 4:42am, causing minimal damage and no reported injuries. The quake was centered near Oakland and left around 2,000 customers without power.

=== Key Takeaway ===
The key takeaway is that a magnitude 4.2 earthquake struck the San Francisco area, causing minimal damage and no reported injuries, but did leave around 2,000 customers without power.


## Text completion

**Exercise**: In this section adjust the `max_completion_tokens` and `temperature` settings below to obtain different responses. Show some examples with the prompt "Continue the story: It was a great time to be alive" with the model "llama-3.1-8b-instant".

* max_completion_tokens - the maximum number of tokens to generate. Note that longer words are made of multiple tokens (set to 200 and 500)
* temperature (positive number) - the higher the number the more random (creative) the output (set to 0.2, 0.8, 2)

In [16]:
# ANSWER (set max_completion_tokens=200, do not have a temperature setting)
def complete_story(prompt, max_tokens=None, temperature=None):
    params = {
        "messages": [{"role": "user", "content": prompt}],
        "model": "llama-3.1-8b-instant"
    }

    # Add optional params if provided
    if max_tokens is not None:
        params["max_tokens"] = max_tokens
    if temperature is not None:
        params["temperature"] = temperature

    response = client.chat.completions.create(**params)
    return response.choices[0].message.content.strip()
print(complete_story("Continue the story: It was a great time to be alive", max_tokens=200))


It was a great time to be alive, with music filling the streets and laughter echoing off the buildings. The sun was shining, casting a warm glow over the bustling city. People of all ages and backgrounds seemed to be enjoying themselves, whether it was a group of teenagers playing a lively game of street hockey or a couple strolling hand in hand through the park.

As I made my way through the crowds, I couldn't help but feel a sense of excitement and possibility. Everywhere I looked, there were reminders that this was a time of great change and progress. New technologies were emerging, new ideas were being explored, and people were pushing the boundaries of what was thought possible.

I stopped to watch a group of artists performing on a street corner. They were a eclectic mix of musicians, dancers, and visual artists, each one bringing their own unique style and energy to the performance. The music was infectious, and soon the crowd was dancing and singing along.

As I watched, a youn

In [17]:
# ANSWER (set max_completion_tokens=500, do not have a temperature setting)
print(complete_story("Continue the story: It was a great time to be alive", max_tokens=500))

It was a great time to be alive, and for Jack, that sentiment couldn't have been more true. The year was 1967, and the world was bursting with creativity and change. Music blasted from every corner, from the Beatles' psychedelic melodies to the soulful cries of Aretha Franklin. The Summer of Love was in full swing, and cities like San Francisco and New York were magnets for young people seeking freedom, self-expression, and a sense of belonging.

As a free-spirited artist, Jack felt like he was exactly where he was meant to be. He spent his days painting vibrant murals, experimenting with music, and hanging out with a diverse group of friends who shared his passion for life.

One sunny afternoon, Jack strolled down Haight-Ashbury, the epicenter of San Francisco's counterculture. He passed by the iconic Fillmore, where Janis Joplin and Jimi Hendrix were about to take the stage. The air vibrated with anticipation, and the scent of incense and patchouli wafted from the crowds gathered out

In [18]:
# ANSWER (set temperature = 0.2, do not have a max_completion_tokens setting)
print(complete_story("Continue the story: It was a great time to be alive", temperature=0.2))

It was a great time to be alive, and Emily couldn't help but feel a sense of excitement and wonder as she walked through the bustling streets of the city. The year was 2050, and the world was a vastly different place from the one her grandparents had grown up in. Technology had advanced at an exponential rate, and the once-distant dreams of science fiction had become a reality.

As she strolled through the crowded market, Emily's eyes were drawn to the towering skyscrapers that seemed to stretch up to the sky. The buildings were covered in a latticework of holographic advertisements, flashing and pulsing with a kaleidoscope of colors that seemed to dance in the air.

She passed by a group of people huddled around a small, sleek device that was projecting a 3D image of a virtual reality world. The scene was so realistic that Emily could almost smell the virtual coffee and feel the virtual breeze on her skin.

As she continued on her way, Emily noticed a group of people gathered around a

In [19]:
# ANSWER (set temperature = 1, do not have a max_completion_tokens setting)
print(complete_story("Continue the story: It was a great time to be alive", temperature=1))

It was a great time to be alive, especially for Emily. Summer had finally arrived in the small town of Oakdale, and the warm sunshine and long days seemed to bring everyone together. The smell of freshly cut grass and blooming flowers filled the air, and the sound of laughter and children's shouts echoed through the streets.

Emily, a bright-eyed and adventurous 20-year-old, was soaking up every moment of it. She had just finished her second year of college and was enjoying a well-deserved break. She had spent the morning lounging in the park, reading a book and soaking up the sun's warm rays. Now, she was walking home, feeling carefree and content.

As she strolled down the main street of Oakdale, Emily noticed a group of friends gathered outside the local diner. They were sipping on milkshakes and laughing together, and Emily couldn't help but feel a pang of envy. She wished she could join them, but she was running a bit late for her shift at the ice cream parlor.

With a quick wave 

Note what happens when the temperature is set too high!

In [20]:
# ANSWER (set temperature = 2, do not have a max_completion_tokens setting)
print(complete_story("Continue the story: It was a great time to be alive", temperature=2))

As I looked out my sun-speckled window, watching the sunlight dance across the streets below, I couldn't help but feel an overwhelming sense of enthusiasm for life. The city was alive – it hummed, pulsed, and beat with an energy all its own. 

The sounds were endless – chatter, beeping horns, the clinking of coffee against paper cups rising up like sweet jazz riffs into the cool, fresh morning air. My heart lifted as the city began waking up around me. People were smiling – genuine smiles – not just smiles hiding tired exhaustion like I often see people wearing on the weekdays; no this was happiness unadulerated pure pure love in the fresh morning dew. And I know why 

My family was coming over to the tiny diner down by Park street to try a taste of my cousin’s latest invention an all-mushroom breakfast pizza that everyone had told they couldn's miss at the weekend markets. It wasn't about the meal that awaited us at this small place with walls painted by some aspiring artist; it certa

### Zero-shot and one-short prompting for question-answering

This section shows the impact of prompting on the response. Zero-shot prompting means we provide the prompt without any examples or additional context. Let us initially ask Mistral a question using no prompting.

In [21]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "How do two chemicals react?"}],
    temperature = 0.8,
)

Markdown(response.choices[0].message.content)

Chemical reactions between two substances occur when there's a change in their chemical bonds, resulting in the formation of new substances. Let's break down the process:

1. **Chemical Energy**: Two chemicals (reactants) come into contact with each other. This interaction can be triggered by various factors, such as heat, light, or the presence of a catalyst.
2. **Bond Breaking**: The molecules of the reactants vibrate and collide with each other. When they collide, their chemical bonds break, releasing energy.
3. **Bond Forming**: The broken bonds from the reactants are rearranged to form new bonds with other reactants, resulting in the formation of new products.
4. **Energy Release or Absorption**: Chemical reactions often involve the release or absorption of energy, which can be in the form of heat, light, or sound.

**Types of Chemical Reactions**:

1. **Synthesis Reaction**: Two or more reactants combine to form a new compound (e.g., hydrogen + oxygen → water).
2. **Decomposition Reaction**: A single reactant breaks down into two or more products (e.g., water → hydrogen + oxygen).
3. **Single-Displacement Reaction**: One element displaces another element from a compound (e.g., zinc + copper sulfate → zinc sulfate + copper).
4. **Double-Displacement Reaction**: Two compounds exchange partners to form new products (e.g., sodium chloride + silver nitrate → sodium nitrate + silver chloride).

**Factors Affecting Chemical Reactions**:

1. **Concentration**: Higher concentrations can increase reaction rates.
2. **Temperature**: Higher temperatures can increase reaction rates, but excessive heat can slow or stop reactions.
3. **Pressure**: Increased pressure can increase reaction rates, but it depends on the reaction.
4. **Catalysts**: Substances that speed up reactions without being consumed by them.
5. **Surface Area**: Increasing surface area can increase reaction rates.

**Notable Examples**:

1. **Combustion Reaction**: Burning wood or gasoline releases energy (e.g., wood + oxygen → carbon dioxide + water + heat).
2. **Acid-Base Reaction**: Mixing acid and base produces a salt and water (e.g., hydrochloric acid + sodium hydroxide → sodium chloride + water).

Keep in mind that this is a simplified explanation. Chemical reactions can be complex, and factors like entropy and equilibrium can influence the outcome.

Do you have a specific reaction or topic you'd like me to explore further?

**Exercise:** Ask the same question but modify the prompt to return the answer to the same question in a simpler form (still using the llama-3.1-8b-instant model). Experiment with different prompts.

In [22]:
# ANSWER
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Explain how two chemicals react in simple words a 10-year-old could understand."}],
    temperature=0.8,
)
Markdown(response.choices[0].message.content)


Let's say we have two chemicals: Soda and Baking Soda.

When we mix these two chemicals together, we get a fun reaction. Here's what happens:

1. Soda is a type of liquid that's full of tiny particles called atoms.
2. Baking Soda is a type of powder that's also made of tiny particles called atoms.
3. When we mix the Soda and Baking Soda together, the tiny particles start to get excited and move around really fast.
4. As they move around, they start to rub against each other and make a new substance called Carbon Dioxide.
5. The Carbon Dioxide is a gas that gets released into the air as bubbles.
6. When we see the bubbles rising up, it's because the Carbon Dioxide is trying to get out of the mixture.

So, in simple words, when we mix Soda and Baking Soda, they create a reaction that makes bubbles. This reaction is fun to see and hear, and it's a great example of how chemicals can interact with each other.

Here's a simple equation that represents this reaction:

Soda (Hydrogen peroxide) + Baking Soda (Sodium bicarbonate) → Carbon Dioxide (CO2) + Water (H2O)

In [24]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Give a short and simple explanation of how two chemicals react."}],
    temperature=0.8,
)
Markdown(response.choices[0].message.content)



Let's consider a simple example of a chemical reaction between hydrogen gas (H2) and oxygen gas (O2) to form water (H2O).

**Reaction:**

2H2 (hydrogen gas) + O2 (oxygen gas) → 2H2O (water)

**Explanation:**

1. The hydrogen and oxygen molecules collide with each other.
2. The oxygen molecule (O2) breaks apart into individual oxygen atoms.
3. The hydrogen molecules (H2) break apart into individual hydrogen atoms.
4. The hydrogen and oxygen atoms combine to form water molecules (H2O).

This reaction is an example of a combustion reaction, where two different elements combine to form a new compound.

In [25]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "How do two chemicals react? Use a food or cooking example to explain."}],
    temperature=0.8,
)
Markdown(response.choices[0].message.content)


Let's consider a common example from cooking to explain how two chemicals react. A chemical reaction occurs when two or more substances interact to form new substances.

**Example: Rancid Butter and Baking Soda**

Imagine you're making cookies and you add a pinch of baking soda (sodium bicarbonate) to the dough. The baking soda reacts with the acidic properties in the butter (lactic acid) to produce carbon dioxide gas.

The chemical reaction is:

Sodium bicarbonate (NaHCO3) + Lactic acid (CH3CH(OH)COOH) → Carbon dioxide (CO2) + Water (H2O) + Salt (NaCH3CH(OH)COO)

In this reaction, the baking soda (sodium bicarbonate) reacts with the lactic acid in the butter to form carbon dioxide gas, which gets trapped in the dough, making it rise. This reaction is a classic example of an acid-base reaction, where the acid in the butter reacts with the base in the baking soda to form a new compound.

**What's happening at a molecular level?**

When the baking soda comes into contact with the lactic acid in the butter, the sodium and hydrogen ions in the baking soda transfer to the lactic acid molecules. This transfer of ions causes the molecules to break apart, releasing carbon dioxide gas as a byproduct. The carbon dioxide gets trapped in the dough, causing it to rise.

**Key takeaways:**

1. Chemical reactions occur when two or more substances interact to form new substances.
2. Acid-base reactions involve the transfer of ions between molecules, resulting in the formation of new compounds.
3. The reaction between baking soda and lactic acid in butter is a classic example of an acid-base reaction, producing carbon dioxide gas and causing the dough to rise.

I hope this example helps illustrate how two chemicals react!

### One-shot prompting ###

Next, note the dramatic change when we give the following template setting a new role and providing an English question followed by a French translation.

In [26]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "system",
             "content": "You translate English to French."},
              {"role": "user",
               "content": "What time is it?"},
               {"role": "assistant",
               "content": "Quelle heure est-il?"},
              {"role": "user",
               "content": "How do two chemicals react?"}],
    temperature = 0.8,
)
print(response.choices[0].message.content)

Comment deux produits chimiques réagissent-ils?


### Few-shot prompting

Recall that since the text generation process outputs one token at a time, their outputs often need adjusting. This is where examples can help.

In [27]:
prompt1 = "I'm gonna head out now, see you later."
response1 = "I will be leaving now. See you later."

prompt2 =  "That movie was super cool!"
response2 = "The movie was very impressive."

prompt3 = "Can't make it to the meeting, sorry."


response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[
        {"role": "system", "content": "You are a professional editor. Rewrite casual sentences into a formal tone."},
        {"role": "user", "content": prompt1},
        {"role": "assistant", "content": response1},
        {"role": "user", "content": prompt2},
        {"role": "assistant", "content": response2},
        {"role": "user", "content": prompt3},
    ]
)

print(response.choices[0].message.content.strip())


Regrettably, I will be unable to attend the meeting. My apologies.


The output can also be moulded to provide SQL output.

In [28]:
prompt1 = "Show me all users who signed up in the last 30 days."
response1 = "SELECT * FROM users WHERE signup_date >= CURRENT_DATE - INTERVAL '30 days';"

prompt2 = "What is the average order value?"
response2 =  "SELECT AVG(order_total) FROM orders;"

prompt3 = "List products that are out of stock."

response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[
        {"role": "system", "content": "You are an assistant that translates natural language to SQL."},
        {"role": "user", "content": prompt1},
        {"role": "assistant", "content": response1},
        {"role": "user", "content": prompt2},
        {"role": "assistant", "content": response2},
        {"role": "user", "content": prompt3},
    ]
)

print(response.choices[0].message.content.strip())


SELECT * FROM products WHERE quantity_in_stock = 0;


**Exercise**: Create a few examples to train the "llama3-70b-8192" LLM to take in user content in the form below and provide output as a pandas dataframe. Use the `exec` function to execute its output to display the answer of sample input as a data frame.

Example:

given the user content

"""

| col1 | col2 | col3

| 32 | 27 | 25

| 64 | 23 | 14

"""

train the model to output

df = pd.DataFrame({'col1': [32, 64], 'col2': [27, 23], 'col3': [25, 14]})



In [30]:
#ANSWER
prompt_example = """
You are a Python assistant. Convert tabular text input into a pandas DataFrame definition.

Input:
| name | age | city |
| John | 30  | NYC  |
| Lisa | 25  | LA   |

Output:
df = pd.DataFrame({'name': ['John', 'Lisa'], 'age': [30, 25], 'city': ['NYC', 'LA']})
"""

# New user content to test
user_input = """
| col1 | col2 | col3 |
| 32   | 27   | 25   |
| 64   | 23   | 14   |
"""

response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[
        {"role": "system", "content": "You are a Python assistant that converts tables to pandas DataFrames."},
        {"role": "user", "content": prompt_example},
        {"role": "assistant", "content": "df = pd.DataFrame({'name': ['John', 'Lisa'], 'age': [30, 25], 'city': ['NYC', 'LA']})"},
        {"role": "user", "content": user_input}
    ],
    temperature=0.2
)

generated_code = response.choices[0].message.content.strip()
print("Generated code:\n", generated_code)

exec(generated_code)
display(df)


Generated code:
 df = pd.DataFrame({'col1': [32, 64], 'col2': [27, 23], 'col3': [25, 14]})


,col1,col2,col3
0,32,27,25
1,64,23,14


Also show what happens when the question is asked in the absence of a system role and without few-shot prompting.

In [31]:
# ANSWER
user_input = """
| col1 | col2 | col3 |
| 32   | 27   | 25   |
| 64   | 23   | 14   |
"""

response = client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[
        {"role": "user", "content": f"Convert this to a pandas dataframe:\n{user_input}"}
    ],
    temperature=0.2
)

print(response.choices[0].message.content.strip())


Here is the equivalent pandas DataFrame:
```
import pandas as pd

data = {'col1': [32, 64], 'col2': [27, 23], 'col3': [25, 14]}
df = pd.DataFrame(data)

print(df)
```
Output:
```
   col1  col2  col3
0    32    27    25
1    64    23    14
```
Let me know if you have any questions!


### Chain-of-thought prompting

The results of question-answering can also be improved by prompting the LLM to provide intermediate steps.

**Exercise**: Using the following prompts, compare the answers of the "llama3-8b-8192" model (set seed=21). (If this model is no longer available choose a model with relatively few parameters.)

zero_shot_prompt = "How many s's are in the word 'success'?"

chain_of_thought_prompt = "How many s's are in the word 'success'? Explain your answer step by step by going through each letter in turn."

In [32]:
# ANSWER
zero_shot_prompt = "How many s's are in the word 'success'?"
chain_of_thought_prompt = "How many s's are in the word 'success'? Explain your answer step by step by going through each letter in turn."
response_zero_shot = client.chat.completions.create(
    model="llama3-8b-8192",
    messages=[{"role": "user", "content": zero_shot_prompt}],
    seed=21,
    temperature=0.2
)

print("Zero-shot response:\n", response_zero_shot.choices[0].message.content.strip())


Zero-shot response:
 There are 2 s's in the word "success".


In [33]:
response_cot = client.chat.completions.create(
    model="llama3-8b-8192",
    messages=[{"role": "user", "content": chain_of_thought_prompt}],
    seed=21,
    temperature=0.2
)

print("Chain-of-thought response:\n", response_cot.choices[0].message.content.strip())


Chain-of-thought response:
 Let's go through each letter in the word "success" to count the number of s's:

1. S - (first letter)
2. U - (second letter)
3. C - (third letter)
4. C - (fourth letter)
5. E - (fifth letter)
6. S - (sixth letter)
7. S - (seventh letter)

As we go through each letter, we can see that there are two S's in the word "success".


## Comparison of LLMs

**Exercise**: Compare the performance of 2 LLMs by outputting the answers of the following questions into a dataframe.

    "Tell me a joke about data science.",
    "How can one calculate 22 * 13 mentally?",
    "Write a creative story about a baby learning to crawl.",

Column headings:

Model Name | Question | Answer

In [34]:
pd.set_option('display.max_colwidth', None) # allows wide dataframes to be viewed
models = ["gemma2-9b-it", "llama-3.1-8b-instant"] #can edit this

# Questions to ask
questions = [
    "Tell me a joke about data science.",
    "How can one calculate 22 * 13 mentally?",
    "Write a creative story about a baby learning to crawl."
]

# Store results
results = []

# Query each model for each question
for model in models:
    for question in questions:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": question}],
            temperature=0.7  # A bit of creativity but not too wild
        )
        answer = response.choices[0].message.content.strip()
        results.append({
            "Model Name": model,
            "Question": question,
            "Answer": answer
        })

# Convert to DataFrame
df_results = pd.DataFrame(results)

# Show result
df_results


,Model Name,Question,Answer
0,gemma2-9b-it,Tell me a joke about data science.,Why did the data scientist break up with the statistician? \n\nBecause they had too many p-values and not enough real-world impact! 😂 \n\n\nLet me know if you'd like to hear another one! 😄
1,gemma2-9b-it,How can one calculate 22 * 13 mentally?,"Here's a way to calculate 22 * 13 mentally:\n\n**1. Break it down:**\n\n* Think of 22 as (20 + 2).\n\n**2. Distribute:**\n\n* Now you have (20 + 2) * 13 \n* Multiply 20 by 13: 20 * 13 = 260\n* Multiply 2 by 13: 2 * 13 = 26\n\n**3. Add the results:**\n\n* 260 + 26 = 286\n\n\n**Therefore, 22 * 13 = 286**"
2,gemma2-9b-it,Write a creative story about a baby learning to crawl.,"Bartholomew Buckleberry III, a cherub-cheeked infant with a shock of ginger hair, surveyed his kingdom. It was a vast expanse of plush carpet, striped with the fearsome shadows of toy giraffes and a rogue, purple dinosaur. The world was a thrilling, mysterious place, but Bartholomew was hopelessly stranded. He longed to explore, to conquer the mountainous pillows and the treacherous gap between his crib and the soft, beckoning rug.\n\nHe had watched, with envious eyes, his big sister, Penelope, navigate this terrain with ease. She'd zoom across the room like a furry comet, leaving a trail of scattered toys in her wake. Bartholomew wanted to be just like Penelope. He wanted to rumble, to roll, to conquer!\n\nOne afternoon, Bartholomew felt a surge of determination. He stretched his chubby arms, wiggling his fingers like nascent claws. He pushed with his legs, his tiny fists clenching and unclenching on the plush carpet. He wobbled, he teetered, he almost toppled over, but he didn't give up.\n\nSuddenly, a sensation of movement, of momentum, took over. Bartholomew was... crawling! He propelled himself forward with a jerky, uneven motion, his face scrunched in concentration. He felt a thrill of triumph as he reached the edge of his crib, then another as he crossed the perilous gap.\n\nThe world exploded in a cacophony of colors and textures. Bartholomew reached out a chubby hand, his fingers brushing against the plush fur of the giraffe. He giggled, a sound like a tiny wind chime, and pulled himself closer to the towering beast.\n\nThe giraffe, once a menacing shadow, now became a plaything. Bartholomew grabbed its long neck, rocking it back and forth, his contented smile widening. He had conquered the unknown, he had navigated the treacherous landscape, and he had discovered a whole new world of adventure.\n\nFrom that day on, Bartholomew Buckleberry III was a changed baby. He crawled with newfound confidence, his tiny body a blur of motion across the room. He wasn't just Bartholomew the baby anymore. He was Bartholomew the Explorer, the Conqueror, the Crawling King. And his kingdom, once a place of limitations, now stretched before him, ripe for the taking."
3,llama-3.1-8b-instant,Tell me a joke about data science.,Why did the data scientist quit his job?\n\nBecause he couldn't visualize a future in the company.
4,llama-3.1-8b-instant,How can one calculate 22 * 13 mentally?,"To calculate 22 * 13 mentally, you can use the following method:\n\n1. Break down the multiplication into easier parts: \n22 * 13 can be rewritten as (20 + 2) * 13.\n\n2. Multiply 20 and 13: \n20 * 13 = 260\n\n3. Multiply 2 and 13: \n2 * 13 = 26\n\n4. Add the results from step 2 and step 3: \n260 + 26 = 286\n\nTherefore, 22 * 13 equals 286."
5,llama-3.1-8b-instant,Write a creative story about a baby learning to crawl.,"**The Great Crawling Adventure**\n\nLily was a little ball of curiosity, with a mop of soft brown hair and a smile that could light up a room. At six months old, she was bursting with energy and a desire to explore the world around her. Her mom, Emma, had been patiently waiting for this moment – the moment when Lily would finally learn to crawl.\n\nOne sunny morning, Emma placed Lily on her tummy on the soft carpet of their living room. ""Go get it, baby gi

### Bonus

See if you can prompt an LLM to perform sentiment analysis (output 'Positive' or 'Negative' only) on a given piece of text.

In [35]:
# ANSWER
text = "I absolutely love how easy this software is to use!"

prompt = f"""
Classify the sentiment of the following review as either Positive or Negative.
Only respond with 'Positive' or 'Negative'.

Review: "{text}"
Sentiment:
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0,  # Low temperature to reduce randomness
)

sentiment = response.choices[0].message.content.strip()
print("Predicted Sentiment:", sentiment)


Predicted Sentiment: Positive


## Conclusion

We worked with a few Large Language Models (LLMs) using Groq and experimented with prompting for summarisation, text completion and question-answering tasks.

We also explored controlling the randomness (creativity) of output through the temperature setting and tried different types of prompting to achieve desired forms of output.

## References
1. [Groq's prompting guide](https://console.groq.com/docs/prompting)
2. [Groq's playground](https://console.groq.com/playground)



---



---



> > > > > > > > > © 2025 Institute of Data


---



---



